# 04. Carga: dados do IBGE no Postgres

Leva para o banco `dengue-eda` a população baixada no `03_coleta_ibge.ipynb` e cria tabelas de apoio com os
estados e municípios.

| Tabela | Conteúdo | Tipo |
|---|---|---|
| `populacao_ne` | população por município do Nordeste e ano, como veio do IBGE | bruta |
| `estado` | código, sigla, nome e região dos 27 estados | apoio |
| `municipio` | código, nome, estado e região de todos os municípios do Brasil | apoio |
| `populacao_municipio_ano` | população pronta para análise, com 2007 e 2023 preenchidos | tratada |

Para análises e dashboard, use `populacao_municipio_ano` e cruze com `municipio` e `estado` pelo código.

**Neste notebook**
1. Conexão
2. Carga
3. Conferência

In [ ]:
import sys
from pathlib import Path
RAIZ = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "config.py").exists())
sys.path.append(str(RAIZ / "src"))

import pandas as pd
import psycopg
from carga.ibge import carga_ibge
from config import BANCO

def consulta(q):
    with psycopg.connect(**BANCO) as con:
        cur = con.execute(q)
        return pd.DataFrame(cur.fetchall(), columns=[d.name for d in cur.description])

## 1. Conexão
Banco Postgres local. Os dados de acesso ficam no arquivo `.env` na raiz do projeto (modelo em `.env.example`).

In [ ]:
consulta("select version()")

## 2. Carga
Lê o arquivo da população, baixa estados e municípios da API de Localidades do IBGE e grava as 4 tabelas
(código em `src/carga/ibge/carga_ibge.py`):

- **`populacao_ne`**: o arquivo como veio, com a população como número inteiro.
- **`estado`** e **`municipio`**: todos os municípios do Brasil, com duas divisões regionais do IBGE: a antiga
  (microrregião e mesorregião) e a atual, de 2017 (região imediata e intermediária).
- **`populacao_municipio_ano`**: feita no banco a partir de `populacao_ne`, sem as linhas vazias e com
  **2007 e 2023 preenchidos** pela média do ano anterior e do seguinte (marcados como `interpolada`).

Tudo é refeito de uma vez: se der erro no meio, nada muda. Pode rodar de novo quando quiser.

In [ ]:
carga_ibge.carregar()

## 3. Conferência

### 3.1 População bruta
O banco deve ter as mesmas linhas do arquivo. A partir de 2008 são 1794 municípios; antes há um pouco menos
(municípios criados depois). 2007 e 2023 não existem nesta tabela.

In [ ]:
pop = carga_ibge.ler_populacao()
n_banco = consulta("select count(*) as n from populacao_ne")["n"][0]
print(f"Linhas: arquivo {len(pop):,} | banco {n_banco:,} | {'iguais' if n_banco == len(pop) else 'DIFERENTES'}")
consulta("""
    select ano, count(populacao) as municipios_com_populacao, sum(populacao) as populacao_ne
    from populacao_ne group by ano order by ano""").set_index("ano")

### 3.2 Estados e municípios
Todos os municípios da população precisam existir em `municipio` (esperado: 0 faltando).

In [ ]:
faltando = consulta("""
    select count(distinct p.cod_mun7) as n from populacao_ne p
    left join municipio m using (cod_mun7) where m.cod_mun7 is null""")["n"][0]
print("Municípios da população sem cadastro:", faltando)
consulta("""
    select e.regiao, count(*) as municipios
    from municipio m join estado e on e.uf_cod = m.uf_cod
    group by e.regiao order by e.regiao""")

In [ ]:
# exemplo: cruzar as tabelas a partir só do código (população de 2024 no RN)
consulta("""
    select e.sigla, m.nome as municipio, m.mesorregiao, p.populacao
    from populacao_municipio_ano p
    join municipio m using (cod_mun7)
    join estado e on e.uf_cod = m.uf_cod
    where p.ano = 2024 and e.sigla = 'RN'
    order by p.populacao desc limit 10""")

### 3.3 População tratada
Esperado: todos os anos de 2000 a 2026, 1794 municípios a partir de 2008 e nenhum ano faltando no meio da série.

In [ ]:
buracos = consulta("""
    select count(*) as n from (
        select cod_mun7 from populacao_municipio_ano
        group by cod_mun7 having max(ano) - min(ano) + 1 > count(*)) t""")["n"][0]
print("Municípios com ano faltando no meio da série:", buracos)
consulta("""
    select ano, count(*) as municipios, sum(populacao) as populacao_ne, string_agg(distinct fonte, ', ') as fonte
    from populacao_municipio_ano group by ano order by ano""").set_index("ano")

In [ ]:
# exemplo: Natal, com 2007 e 2023 preenchidos
consulta("""
    select ano, populacao, fonte from populacao_municipio_ano
    where cod_mun7 = '2408102' and (ano between 2005 and 2009 or ano between 2021 and 2025)
    order by ano""").set_index("ano")